# Day 3: Exploratory Data Analysis (EDA)

Bluestock Mutual Fund Analytics EDA notebook. The analysis follows the Day 3 task order and uses cleaned datasets from `data/processed/`. Exported report charts are stored in `reports/day3_charts/`.

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid', context='notebook')
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = BASE_DIR / 'data' / 'processed'
CHART_DIR = BASE_DIR / 'reports' / 'day3_charts'


## Task 1: NAV Trend Analysis

Daily NAV trends were analysed for all 40 schemes from 2022 to 2026. The exported charts highlight the 2023 bull-run period and a 2024 correction window.

![NAV trend all schemes](../reports/day3_charts/task_01_nav_trend_all_40_schemes.png)

![Average indexed NAV](../reports/day3_charts/task_01_nav_average_indexed.png)

![Equity indexed NAV](../reports/day3_charts/task_01_nav_equity_indexed.png)

![2024 correction focus](../reports/day3_charts/task_01_nav_2024_correction_focus.png)

In [12]:
fund = pd.read_csv(DATA_DIR / '01_fund_master.csv')
nav = pd.read_csv(DATA_DIR / '02_nav_history.csv', parse_dates=['date'])
nav_merged = nav.merge(fund[['amfi_code', 'scheme_name', 'category']], on='amfi_code')
# Plotly workflow for main chart:
# px.line(nav_merged, x='date', y='nav', color='scheme_name', title='Daily NAV Trend - All 40 Schemes')


## Task 2: AUM Growth by Fund House

AUM was grouped by AMC and year from 2022 to 2025. SBI Mutual Fund is highlighted as the 2025 leader at Rs. 12.50 lakh crore.

![AUM growth by AMC](../reports/day3_charts/task_02_aum_growth_by_amc.png)

![AUM 2025 market share](../reports/day3_charts/support_aum_2025_market_share.png)

In [13]:
aum = pd.read_csv(DATA_DIR / '03_aum_by_fund_house.csv', parse_dates=['date'])
aum['year'] = aum['date'].dt.year
yearly_aum = aum.sort_values('date').groupby(['year', 'fund_house'], as_index=False).tail(1)
# Seaborn workflow:
# sns.barplot(data=yearly_aum, x='fund_house', y='aum_lakh_crore', hue='year')


## Task 3: SIP Inflow Time-Series

Monthly SIP inflows were analysed from January 2022 to December 2025. December 2025 is annotated as the all-time high in the available dataset at Rs. 31,002 crore.

![SIP inflow trend](../reports/day3_charts/task_03_sip_inflow_trend.png)

![Active SIP accounts](../reports/day3_charts/support_active_sip_accounts.png)

In [14]:
sip = pd.read_csv(DATA_DIR / '04_monthly_sip_inflows.csv')
sip['month_dt'] = pd.to_datetime(sip['month'] + '-01')
# Plotly workflow:
# fig = px.line(sip, x='month_dt', y='sip_inflow_crore', markers=True)


## Task 4: Category Inflow Heatmap

Monthly category inflows were pivoted into a category-by-month heatmap. The available category window is April 2024 to March 2025.

![Category inflow heatmap](../reports/day3_charts/task_04_category_inflow_heatmap.png)

In [15]:
category = pd.read_csv(DATA_DIR / '05_category_inflows.csv')
category_matrix = category.pivot(index='category', columns='month', values='net_inflow_crore')
# Seaborn workflow:
# sns.heatmap(category_matrix, cmap='YlOrRd')


## Task 5: Investor Demographics

Investor demographics were analysed through age distribution, SIP ticket-size distribution by age group, and gender split.

![Age group distribution](../reports/day3_charts/task_05_age_group_distribution_pie.png)

![SIP amount boxplot by age](../reports/day3_charts/task_05_sip_amount_box_by_age.png)

![Gender split](../reports/day3_charts/task_05_gender_split.png)

In [16]:
transactions = pd.read_csv(DATA_DIR / '08_investor_transactions.csv', parse_dates=['transaction_date'])
sip_transactions = transactions[transactions['transaction_type'] == 'SIP']
# Age pie, SIP amount boxplot by age group, and gender split are generated from transactions.


## Task 6: Geographic Distribution

Geographic SIP distribution was analysed by state and city tier. The task uses SIP transaction amounts as the geography metric.

![State SIP distribution](../reports/day3_charts/task_06_state_sip_distribution_bar.png)

![City tier SIP split](../reports/day3_charts/task_06_city_tier_pie.png)

In [17]:
state_sip = sip_transactions.groupby('state')['amount_inr'].sum().sort_values(ascending=False)
city_tier_sip = sip_transactions.groupby('city_tier')['amount_inr'].sum()


## Task 7: Folio Count Growth

Industry folio growth was analysed from January 2022 to December 2025, including the latest category-wise folio mix.

![Folio count growth](../reports/day3_charts/task_07_folio_count_growth.png)

![Folio mix Dec 2025](../reports/day3_charts/support_folio_mix_dec_2025.png)

In [18]:
folios = pd.read_csv(DATA_DIR / '06_industry_folio_count.csv')
folios['month_dt'] = pd.to_datetime(folios['month'] + '-01')


## Task 8: NAV Return Correlation Matrix

Daily NAV returns were computed for the top 10 schemes by AUM and visualised as a pairwise correlation matrix.

![NAV return correlation matrix](../reports/day3_charts/task_08_nav_return_correlation_matrix.png)

In [19]:
performance = pd.read_csv(DATA_DIR / '07_scheme_performance.csv')
selected_codes = performance.sort_values('aum_crore', ascending=False).head(10)['amfi_code']
# Pivot NAV by selected AMFI codes, compute pct_change(), then corr(); visualize with sns.heatmap.


## Task 9: Sector Allocation Donut

Portfolio holdings were aggregated across equity funds to estimate sector allocation.

![Sector allocation donut](../reports/day3_charts/task_09_sector_allocation_donut.png)

In [20]:
holdings = pd.read_csv(DATA_DIR / '09_portfolio_holdings.csv')
equity_codes = fund[fund['category'] == 'Equity']['amfi_code']
sector_weights = holdings[holdings['amfi_code'].isin(equity_codes)].groupby('sector')['weight_pct'].sum()


## Supporting Performance Diagnostics

Additional charts were included to exceed the 15-chart requirement and enrich the EDA story.

![Top 10 3-year returns](../reports/day3_charts/support_top_10_3yr_returns.png)

![Top 10 Sharpe ratio](../reports/day3_charts/support_top_10_sharpe_ratio.png)

## Task 10: Key EDA Findings

1. SBI Mutual Fund is the largest AMC in 2025 with Rs. 12.50 lakh crore AUM, well ahead of peers.
2. Monthly SIP inflows grew 169.2% from Jan 2022 to Dec 2025, reaching the dataset high of Rs. 31,002 crore.
3. Liquid recorded the highest cumulative category net inflow at Rs. 451,275 crore during the available category window.
4. The 26-35 age group contributes the largest share of investor transaction records.
5. Madhya Pradesh is the leading SIP state by amount with Rs. 2.1 crore in SIP transactions.
6. Industry folios increased by 12.86 crore, from 13.26 crore to 26.12 crore.
7. The selected top 10 funds show an average pairwise daily return correlation of 0.01, indicating limited co-movement in this synthetic NAV sample.
8. Banking is the largest aggregated equity portfolio sector at 19.2% of summed sector weights.
9. The 2024 correction focus chart shows the average indexed NAV local low at 2024-07-02.
10. December 2025 also has the largest active SIP account base in the dataset at 9.35 crore accounts.

## Deliverable Checklist

- `EDA_Analysis.ipynb` contains all 10 Day 3 task sections.
- `reports/day3_charts/` contains 20 exported PNG charts.
- `reports/EDA_Findings_summary.md` contains the final findings summary.
- The notebook follows the task order from the Day 3 instruction sheet.